# 从 Token 到 Loss：端到端 Transformer 源码实验

这个 Notebook 不依赖外部 Tokenizer、预训练模型或数据集。我们用极小 Shape 构造一个**正确区分源序列与目标序列**的 Encoder–Decoder Transformer，并跟踪：

`Token IDs → Embedding/PE → Encoder → Decoder Self-Attention → Cross-Attention → Logits → Loss → Backward`

示例固定 `B=2, S_src=6, S_tgt=5, H=8, N=2, D=4, V=12`。代码解释直接写在对应操作旁边。

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
torch.set_printoptions(precision=3, sci_mode=False)

@dataclass
class Config:
    vocab_size: int = 12      # V：词表大小
    hidden_size: int = 8      # H：每个 Token 的隐藏向量维度
    num_heads: int = 2        # N：Attention 头数
    ffn_size: int = 16        # F：MLP 中间维度
    max_len: int = 16
    num_layers: int = 1
    dropout: float = 0.0      # 为了让重复运行结果稳定，实验关闭 Dropout

cfg = Config()
assert cfg.hidden_size % cfg.num_heads == 0
print('每头维度 D = H / N =', cfg.hidden_size // cfg.num_heads)

## 1. 构造源序列、右移后的目标输入和标签

- `src_ids` 送入 Encoder；
- `tgt_in` 以 `BOS` 开头，送入 Decoder；
- `labels` 是 `tgt_in` 向左移动一位后的正确答案；
- 两个源句子补齐到长度 6，目标句子补齐到长度 5；
- 源、目标长度故意不同，使 Cross-Attention 得到 `[B,N,5,6]`。

In [ ]:
PAD, BOS, EOS = 0, 1, 2

# 两条源序列，Shape=[B,S_src]=[2,6]。0 是补齐用的 PAD。
src_ids = torch.tensor([
    [BOS, 3, 4, 5, EOS, PAD],
    [BOS, 6, 7, EOS, PAD, PAD],
])

# Teacher Forcing：Decoder 输入以 BOS 开始。
tgt_in = torch.tensor([
    [BOS, 8, 9, 5, PAD],
    [BOS, 10, 5, PAD, PAD],
])

# 每个位置预测下一个 Token；PAD 标签稍后通过 ignore_index 排除。
labels = torch.tensor([
    [8, 9, 5, EOS, PAD],
    [10, 5, EOS, PAD, PAD],
])

src_valid = src_ids.ne(PAD)  # [B,S_src]，True 表示真实 Token
tgt_valid = tgt_in.ne(PAD)   # [B,S_tgt]

print('src_ids :', tuple(src_ids.shape))
print('tgt_in  :', tuple(tgt_in.shape))
print('labels  :', tuple(labels.shape))
print('src_valid:\n', src_valid)

## 2. Embedding 与正余弦位置编码

Embedding 完成 `[B,S] → [B,S,H]`。位置编码与 Embedding 相加，不改变 Shape。

公式：$PE(pos,2i)=\sin(pos\cdot10000^{-2i/H})$，$PE(pos,2i+1)=\cos(pos\cdot10000^{-2i/H})$。

In [ ]:
class SinusoidalPositionEncoding(nn.Module):
    def __init__(self, hidden_size: int, max_len: int):
        super().__init__()
        assert hidden_size % 2 == 0, '本实验使用偶数 hidden_size'

        pe = torch.zeros(max_len, hidden_size)                    # [max_len,H]
        position = torch.arange(max_len).float().unsqueeze(1)    # [max_len,1]
        frequency = torch.exp(
            torch.arange(0, hidden_size, 2).float()
            * (-math.log(10000.0) / hidden_size)
        )                                                       # [H/2]

        # [max_len,1] * [H/2] 通过广播得到 [max_len,H/2]。
        pe[:, 0::2] = torch.sin(position * frequency)
        pe[:, 1::2] = torch.cos(position * frequency)

        # 位置编码不是训练参数，但应随模型一起移动到 CPU/GPU，所以注册为 buffer。
        self.register_buffer('pe', pe.unsqueeze(0))             # [1,max_len,H]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # [B,S,H] + [1,S,H]：同一套位置编码广播给 Batch 中所有样本。
        return x + self.pe[:, :x.size(1)]

demo_embedding = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
demo_position = SinusoidalPositionEncoding(cfg.hidden_size, cfg.max_len)
demo_src_x = demo_position(demo_embedding(src_ids))
print('Embedding + PE:', tuple(demo_src_x.shape))

## 3. 多头注意力：支持 $S_q \neq S_{kv}$ 的正确实现

核心公式：

$$Q=X_qW^Q,\quad K=X_kW^K,\quad V=X_vW^V$$

$$P=Softmax\left(\frac{QK^T}{\sqrt D}+Mask\right)$$

$$Output=Concat(PV)W^O$$

这份实现分别读取 `S_q` 和 `S_kv`，因此既能处理 Self-Attention，也能处理源、目标长度不同的 Cross-Attention。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert hidden_size % num_heads == 0
        self.hidden_size = hidden_size                         # H
        self.num_heads = num_heads                             # N
        self.head_dim = hidden_size // num_heads               # D=H/N

        # Linear 只变换最后一维；输入输出都是 H，因此投影后仍是 [B,S,H]。
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.out_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        B, S, _ = x.shape
        # [B,S,H] -> [B,S,N,D] -> [B,N,S,D]
        return x.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        attn_mask: torch.Tensor | None = None,
        key_padding_mask: torch.Tensor | None = None,
    ):
        B, S_q, _ = query.shape
        _, S_kv, _ = key.shape

        # 函数参数是投影前的 Hidden State；以下 q/k/v 才对应公式中的 Q/K/V。
        q = self._split_heads(self.q_proj(query))               # [B,N,S_q,D]
        k = self._split_heads(self.k_proj(key))                 # [B,N,S_kv,D]
        v = self._split_heads(self.v_proj(value))               # [B,N,S_kv,D]

        # [B,N,S_q,D] @ [B,N,D,S_kv] -> [B,N,S_q,S_kv]
        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        if attn_mask is not None:
            # attn_mask=[S_q,S_kv]，True=允许查看；广播到所有 Batch 和 Head。
            scores = scores.masked_fill(~attn_mask[None, None, :, :], float('-inf'))

        if key_padding_mask is not None:
            # key_padding_mask=[B,S_kv]，只遮 Key 列，不让 PAD 被任何 Query 读取。
            scores = scores.masked_fill(
                ~key_padding_mask[:, None, None, :],
                float('-inf'),
            )

        # 每个 Query 沿最后的 Key 维归一化。转 FP32 可提升指数与求和稳定性。
        weights = F.softmax(scores.float(), dim=-1).to(q.dtype)  # [B,N,S_q,S_kv]
        weights = self.dropout(weights)

        # [B,N,S_q,S_kv] @ [B,N,S_kv,D] -> [B,N,S_q,D]
        context = weights @ v

        # 合头：[B,N,S_q,D] -> [B,S_q,N,D] -> [B,S_q,H]。
        context = context.transpose(1, 2).contiguous().view(B, S_q, self.hidden_size)
        output = self.out_proj(context)                         # [B,S_q,H]
        return output, weights

## 4. 用 Attention、LayerNorm、残差和 MLP 组装 Encoder/Decoder

本实验使用 Pre-Norm：`x = x + sublayer(norm(x))`。残差支路始终保留归一化前的 `x`。

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, hidden_size: int, ffn_size: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, ffn_size),   # [B,S,H] -> [B,S,F]
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_size, hidden_size),   # [B,S,F] -> [B,S,H]
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.hidden_size)
        self.self_attn = MultiHeadAttention(cfg.hidden_size, cfg.num_heads, cfg.dropout)
        self.norm2 = nn.LayerNorm(cfg.hidden_size)
        self.ffn = FeedForward(cfg.hidden_size, cfg.ffn_size, cfg.dropout)

    def forward(self, x, src_valid):
        norm_x = self.norm1(x)
        attn_out, weights = self.self_attn(
            norm_x, norm_x, norm_x,             # 同源，所以是 Self-Attention
            key_padding_mask=src_valid,
        )
        x = x + attn_out                         # 原始 x 走残差支路
        x = x + self.ffn(self.norm2(x))
        return x, weights


class DecoderLayer(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.hidden_size)
        self.self_attn = MultiHeadAttention(cfg.hidden_size, cfg.num_heads, cfg.dropout)
        self.norm2 = nn.LayerNorm(cfg.hidden_size)
        self.cross_attn = MultiHeadAttention(cfg.hidden_size, cfg.num_heads, cfg.dropout)
        self.norm3 = nn.LayerNorm(cfg.hidden_size)
        self.ffn = FeedForward(cfg.hidden_size, cfg.ffn_size, cfg.dropout)

    def forward(self, x, encoder_output, causal_mask, tgt_valid, src_valid):
        norm_x = self.norm1(x)
        self_out, self_weights = self.self_attn(
            norm_x, norm_x, norm_x,
            attn_mask=causal_mask,               # 不能看未来
            key_padding_mask=tgt_valid,          # 不能读取目标 PAD
        )
        x = x + self_out

        norm_x = self.norm2(x)
        cross_out, cross_weights = self.cross_attn(
            query=norm_x,                        # Q 来自 Decoder
            key=encoder_output,                  # K 来自 Encoder
            value=encoder_output,                # V 来自 Encoder
            key_padding_mask=src_valid,          # 不能读取源 PAD
        )
        x = x + cross_out
        x = x + self.ffn(self.norm3(x))
        return x, self_weights, cross_weights

## 5. 组装完整模型：入口应当先读这里

阅读源码时先看 `forward`：两组 Token 分别进入 Embedding；Encoder 输出作为 Decoder Cross-Attention 的 K/V；Decoder 输出经过 LM Head 得到词表 Logits。

In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.src_embedding = nn.Embedding(cfg.vocab_size, cfg.hidden_size, padding_idx=PAD)
        self.tgt_embedding = nn.Embedding(cfg.vocab_size, cfg.hidden_size, padding_idx=PAD)
        self.position = SinusoidalPositionEncoding(cfg.hidden_size, cfg.max_len)
        self.encoder_layers = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.num_layers)])
        self.encoder_norm = nn.LayerNorm(cfg.hidden_size)
        self.decoder_norm = nn.LayerNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size)

    def forward(self, src_ids, tgt_ids):
        src_valid = src_ids.ne(PAD)                           # [B,S_src]
        tgt_valid = tgt_ids.ne(PAD)                           # [B,S_tgt]
        B, S_src = src_ids.shape
        _, S_tgt = tgt_ids.shape

        src_x = self.position(self.src_embedding(src_ids))    # [B,S_src,H]
        tgt_x = self.position(self.tgt_embedding(tgt_ids))    # [B,S_tgt,H]

        encoder_weights = None
        for layer in self.encoder_layers:
            src_x, encoder_weights = layer(src_x, src_valid)
        encoder_output = self.encoder_norm(src_x)             # [B,S_src,H]

        # 第 i 行只允许查看 0..i 列。Shape=[S_tgt,S_tgt]。
        causal_mask = torch.tril(
            torch.ones(S_tgt, S_tgt, dtype=torch.bool, device=tgt_ids.device)
        )

        decoder_self_weights = cross_weights = None
        for layer in self.decoder_layers:
            tgt_x, decoder_self_weights, cross_weights = layer(
                tgt_x, encoder_output, causal_mask, tgt_valid, src_valid
            )
        decoder_output = self.decoder_norm(tgt_x)             # [B,S_tgt,H]
        logits = self.lm_head(decoder_output)                 # [B,S_tgt,V]

        trace = {
            'src_ids': tuple(src_ids.shape),
            'tgt_ids': tuple(tgt_ids.shape),
            'src_embedding_plus_pe': (B, S_src, self.cfg.hidden_size),
            'tgt_embedding_plus_pe': (B, S_tgt, self.cfg.hidden_size),
            'encoder_self_weights': tuple(encoder_weights.shape),
            'encoder_output': tuple(encoder_output.shape),
            'decoder_self_weights': tuple(decoder_self_weights.shape),
            'decoder_cross_weights': tuple(cross_weights.shape),
            'decoder_output': tuple(decoder_output.shape),
            'logits': tuple(logits.shape),
        }
        attention = {
            'causal_mask': causal_mask,
            'encoder': encoder_weights,
            'decoder_self': decoder_self_weights,
            'cross': cross_weights,
        }
        return logits, trace, attention

## 6. 跑一次完整前向传播并核对 Shape 账本

In [ ]:
model = TinyTransformer(cfg)
logits, trace, attention = model(src_ids, tgt_in)

for name, shape in trace.items():
    print(f'{name:28s} {shape}')

assert trace['encoder_self_weights'] == (2, 2, 6, 6)
assert trace['decoder_self_weights'] == (2, 2, 5, 5)
assert trace['decoder_cross_weights'] == (2, 2, 5, 6)
assert trace['logits'] == (2, 5, 12)
print('\n所有关键 Shape 校验通过。')

## 7. 从 Logits 到 Cross Entropy Loss，再反向传播

`cross_entropy` 内部包含 Log-Softmax。先把 `[B,S_tgt,V]` 展平为 `[B×S_tgt,V]`；`ignore_index=PAD` 让补齐位置不参与 Loss。

In [ ]:
loss = F.cross_entropy(
    logits.reshape(-1, cfg.vocab_size),
    labels.reshape(-1),
    ignore_index=PAD,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
optimizer.zero_grad()
loss.backward()

grad_norm = model.lm_head.weight.grad.norm().item()
before = model.lm_head.weight.detach().clone()
optimizer.step()
changed = not torch.equal(before, model.lm_head.weight.detach())

print('loss:', float(loss))
print('lm_head 梯度范数:', grad_norm)
print('optimizer.step 后参数是否变化:', changed)
assert torch.isfinite(loss)
assert grad_norm > 0
assert changed

## 8. 直接观察 Mask 是否真正生效

下面检查 Decoder 第一个样本、第 0 个头。主对角线上方对应未来位置，应全部为 0；Cross-Attention 的源 PAD 列也应全部为 0。

In [ ]:
causal_mask = attention['causal_mask']
decoder_head0 = attention['decoder_self'][0, 0].detach()
cross_head0 = attention['cross'][0, 0].detach()

print('因果 Mask（1=可见，0=未来）：\n', causal_mask.int())
print('\nDecoder Self-Attention，第 0 个头：\n', decoder_head0)
print('\nCross-Attention，第 0 个头：\n', cross_head0)

future_values = decoder_head0.masked_select(~causal_mask)
assert torch.allclose(future_values, torch.zeros_like(future_values))

# 第一个源样本最后一列是 PAD，对所有目标 Query 的权重都必须为 0。
assert torch.allclose(cross_head0[:, -1], torch.zeros_like(cross_head0[:, -1]))
print('\n因果 Mask 与源 Padding Mask 校验通过。')

## 9. 与 FA/FIA 的连接：Score 中间矩阵为什么昂贵

本实验中的普通 PyTorch Attention 显式生成 `[B,N,S_q,S_kv]` 的 Score 和 Probability。下面只估算这一个张量的容量；实际前向、反向还会有更多中间量。

In [ ]:
def score_memory_mib(B, N, S_q, S_kv, bytes_per_element=2):
    elements = B * N * S_q * S_kv
    return elements, elements * bytes_per_element / 1024**2

for S in [128, 1024, 4096, 16384]:
    elements, mib = score_memory_mib(B=1, N=32, S_q=S, S_kv=S)
    print(f'S={S:5d}  Score 元素={elements:12,d}  BF16容量={mib:9.2f} MiB')

print('\nFA/FIA 保持数学目标不变，核心是分块、融合与 Online Softmax，避免完整大矩阵反复落到外部显存。')